In [1]:
!pip install ultralytics -q
!pip install roboflow -q
!pip install gradio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.9/86.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.

In [5]:
from ultralytics import YOLO
import ultralytics

import roboflow

import glob

import gradio as gr

import cv2
import numpy as np
import os

from matplotlib import pyplot as plt

import pandas as pd

import torch

from IPython import display
display.clear_output()
from IPython.display import display, Image

HOME = os.getcwd()
PROJECT_NAME = 'window_door_room_type_detection'
MODEL_NAME_1 = 'window_door_detect_model'
MODEL_NAME_2 = 'room_detection_project'
RANDOM_STATE = 70725

In [6]:
ultralytics.checks()

Ultralytics 8.3.165 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 41.9/112.6 GB disk)


In [7]:
def load_dataset():
  rf = roboflow.Roboflow(api_key='3sllSNniQTyalwdteomI')
  project1 = rf.workspace("floorplans-gmxhd").project("auto_multi_annotations")
  dataset1 = project1.version(2).download("yolov11")
  project2 = rf.workspace("project-ux5ds").project("estima_ai")
  dataset2 = project2.version(3).download("yolov11")

  DS_LOC_1 = dataset1.location
  DS_LOC_2 = dataset2.location

  return DS_LOC_1, DS_LOC_2

In [8]:
DS_LOC_1, DS_LOC_2 = load_dataset()

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to auto_multi_annotations-2 in yolov11:: 100%|██████████| 19410/19410 [00:03<00:00, 5939.87it/s]


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Estima_ai-3 in yolov11:: 100%|██████████| 5158/5158 [00:01<00:00, 5053.35it/s]


In [9]:
def train_model(DS_LOC_1, DS_LOC_2):

    os.makedirs(os.path.join(HOME, 'models'), exist_ok=True)

    """Обучение модели YOLO"""
    model_1 = YOLO(f'{HOME}/yolo11s.pt')
    model_2 = YOLO(f'{HOME}/yolo11s.pt')

    model_1.train(
    model=os.path.join(HOME, 'yolo11s.pt'),
    data=f"{DS_LOC_1}/data.yaml",
            epochs=100,
            batch=-1,
            device=0 if torch.cuda.is_available() else 'cpu',
            seed=RANDOM_STATE,
            plots=True,
            conf=0.5,
            patience=10,
            project=PROJECT_NAME,
            name=MODEL_NAME_1
            )

    model_1.save(os.path.join(HOME, 'models', f'{MODEL_NAME_1}.pt'))

    model_2.train(
    model=os.path.join(HOME, 'yolo11s.pt'),
    data=f'{DS_LOC_2}/data.yaml',
            epochs=200,
            batch=-1,
            device=0 if torch.cuda.is_available() else 'cpu',
            seed=RANDOM_STATE,
            plots=True,
            conf=0.5,
            patience=10,
            project=PROJECT_NAME,
            name=MODEL_NAME_1
            )
    model_2.save(os.path.join(HOME, 'models', f'{MODEL_NAME_1}.pt'))

In [ ]:
train_model(DS_LOC_1, DS_LOC_2)

100%|██████████| 18.4M/18.4M [00:00<00:00, 214MB/s]


Ultralytics 8.3.165 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=0.5, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/auto_multi_annotations-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=window_door_detect_model, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective

100%|██████████| 755k/755k [00:00<00:00, 19.3MB/s]

Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  3                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  4                  -1  1    103360  ultralytics.nn.modules.block.C3k2            [128, 256, 1, False, 0.25]    
  5                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  6                  -1  1    346112  ultralytics.nn.modules.block.C3k2            [256, 256, 1, True]           
  7                  -1  1   1180672  ultralytics

 22                  -1  1   1511424  ultralytics.nn.modules.block.C3k2            [768, 512, 1, True]           
 23        [16, 19, 22]  1    820182  ultralytics.nn.modules.head.Detect           [2, [128, 256, 512]]          
YOLO11s summary: 181 layers, 9,428,566 parameters, 9,428,550 gradients, 21.6 GFLOPs

Transferred 493/499 items from pretrained weights
Freezing layer 'model.23.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     36/100      7.82G     0.9504     0.5602     0.8954        243        640: 100%|██████████| 297/297 [02:49<00:00,  1.75it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 16/16 [00:09<00:00,  1.64it/s]

                   all        904      15876      0.936       0.85      0.893      0.706



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     37/100      7.52G      0.939     0.5598     0.8982        634        640:  16%|█▋        | 49/297 [00:28<02:27,  1.68it/s]

In [ ]:
def process_image(model_type, image, conf_threshold=0.5):
    img_np = np.array(image)

    if model_type == "Окна и двери":
        # Детекция объектов
        results = window_door_model.predict(img_np, conf=conf_threshold)
        result_img = results[0].plot()
        result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

        # Сбор информации
        detections = []
        for i, box in enumerate(results[0].boxes):
            cls_id = int(box.cls)
            label = window_door_model.names[cls_id]
            conf = float(box.conf)
            bbox = [round(x) for x in box.xyxy[0].tolist()]

            detections.append([
                i+1, label, f"{conf:.2f}",
                bbox[2]-bbox[0], bbox[3]-bbox[1]
            ])

        # Статистика
        windows = sum(1 for d in detections if d[1] == "window")
        doors = sum(1 for d in detections if d[1] == "door")
        summary = f"🪟 Окна: {windows} | 🚪 Двери: {doors}"

        return result_img, summary, detections

    else:
        results = window_door_model.predict(img_np, conf=conf_threshold)
        result_img = results[0].plot()
        result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

        # Сбор информации
        detections = []
        for i, box in enumerate(results[0].boxes):
            cls_id = int(box.cls)
            label = window_door_model.names[cls_id]
            conf = float(box.conf)
            bbox = [round(x) for x in box.xyxy[0].tolist()]

            detections.append([
                i+1, label, f"{conf:.2f}",
                bbox[2]-bbox[0], bbox[3]-bbox[1]
            ])

        # Статистика
        windows = sum(1 for d in detections if d[1] == "window")
        doors = sum(1 for d in detections if d[1] == "door")
        summary = f"🪟 Окна: {windows} | 🚪 Двери: {doors}"

        return result_img, summary, detections